# Stage 01 - Data Ingestion: Ingesting and Parsing Plain Text Files
### Lecture 10: TextLoader, DirectoryLoader, and Encoding Management

This notebook implements production-grade plain text document ingestion:
1. Handling text files with explicit UTF-8 encoding declarations.
2. Parsing single files with `TextLoader`.
3. Batch directory ingestion using `DirectoryLoader` with file pattern filtering.
4. Robust error handling for corrupt encodings and missing files.


In [1]:
# Step 1: Environment Setup and Path Discovery
from pathlib import Path
from langchain_community.document_loaders import TextLoader, DirectoryLoader

# Deterministically locate repository root and text data directory
repo_root = Path.cwd()
while not (repo_root / "data").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

txt_dir = repo_root / "data" / "txt"
print(f"[CONFIG] Plain Text Directory: {txt_dir}")
assert txt_dir.exists(), f"Text directory {txt_dir} must exist."

txt_files = list(txt_dir.glob("*.txt"))
print(f"[INFO] Found {len(txt_files)} text files in directory:")
for f in txt_files:
    print(f"  - {f.name} ({f.stat().st_size} bytes)")

assert len(txt_files) >= 3, "Expected at least 3 sample text files in data/txt."


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_6620\508468689.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader


[CONFIG] Plain Text Directory: D:\Code\AI\rag-mastery\data\txt
[INFO] Found 3 text files in directory:
  - artificial_intelligence.txt (3605 bytes)
  - software_engineering.txt (4397 bytes)
  - web_development.txt (3615 bytes)


In [2]:
# Step 2: Ingesting a Single Text Document with Explicit Encoding
single_file_path = txt_files[0]

# TextLoader requires explicit encoding parameter to prevent OS-dependent codec errors
loader = TextLoader(file_path=str(single_file_path), encoding="utf-8")
loaded_documents = loader.load()

print(f"[INFO] Loaded document count: {len(loaded_documents)}")
primary_doc = loaded_documents[0]
print(f"[INFO] Source: {primary_doc.metadata.get('source')}")
print(f"[INFO] Content Preview ({len(primary_doc.page_content)} characters):")
print(primary_doc.page_content[:150] + "...")

# Assertions
assert len(loaded_documents) == 1, "TextLoader must return exactly one Document for a single text file."
assert len(primary_doc.page_content) > 0, "Document content must not be empty."
assert "source" in primary_doc.metadata, "Metadata must record the file source path."


[INFO] Loaded document count: 1
[INFO] Source: D:\Code\AI\rag-mastery\data\txt\artificial_intelligence.txt
[INFO] Content Preview (3537 characters):
TITLE: Introduction to Artificial Intelligence

AUTHOR: Ahmed Hassan
CATEGORY: Artificial Intelligence
DATE: September 2026

---

1. What is Artificia...


In [3]:
# Step 3: Batch Directory Ingestion with DirectoryLoader
# DirectoryLoader aggregates multiple files matching a glob pattern
dir_loader = DirectoryLoader(
    path=str(txt_dir),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

all_text_docs = dir_loader.load()
print(f"[INFO] Batch DirectoryLoader successfully parsed {len(all_text_docs)} documents.")

for idx, doc in enumerate(all_text_docs, start=1):
    source_name = Path(doc.metadata.get("source", "")).name
    print(f"  Doc {idx}: {source_name} ({len(doc.page_content)} chars)")

# Assertions
assert len(all_text_docs) == len(txt_files), "DirectoryLoader must load all files matching the glob pattern."
assert all(len(d.page_content) > 0 for d in all_text_docs), "All loaded documents must have non-empty content."
print("[SUCCESS] Single and batch plain text ingestion verified.")


[INFO] Batch DirectoryLoader successfully parsed 3 documents.
  Doc 1: artificial_intelligence.txt (3537 chars)
  Doc 2: software_engineering.txt (4256 chars)
  Doc 3: web_development.txt (3514 chars)
[SUCCESS] Single and batch plain text ingestion verified.
